## Notebook grammar

Open in Colab: https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/templates/jaxfne_notebook_template.ipynb

setup -> config -> simulation/probe -> objective cells -> export

This notebook uses package APIs through `import jaxfne as jtfne`; editable inputs are centralized in config cells; readouts are proxy-scoped where named as proxies; exports use JSON/PNG receipts when artifacts are produced.


# JAXFNE Notebook Template

This notebook demonstrates the canonical jaxfne workflow:
configure → construct → simulate → optimize → visualize → export.

## Scope Gates
- **run_status:** `tutorial_scaffold`
- **model_status:** `computational_scaffold`
- **field_solver_status:** `linear_solver`
- **amplitude_status:** `False`

## Setup: Installation & Environment

In [ ]:
# Colab-compatible setup: use local checkout when available; otherwise install jaxfne from current main.
import importlib.util, subprocess, sys
from pathlib import Path
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "jaxfne").is_dir() and (_candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(_candidate))
        break
if importlib.util.find_spec("jaxfne") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jaxfne[viz,opt] @ git+https://github.com/HNXJ/jaxfne.git@main"])


### Setup complete

The next cell starts the tutorial code.


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import jaxfne as jtfne
print(f"jaxfne version: {jtfne.__version__}")


### Imports loaded


In [ ]:
!pip install -q jaxfne
!pip install -q "jaxfne @ git+https://github.com/HNXJ/jaxfne.git@dev"
import os, json, jax
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("JAX_PLATFORM_NAME", os.environ.get("TFNE_BACKEND", "cpu"))
jax.config.update("jax_enable_x64", False)
import jaxfne as jtfne

## Outline

1. **Configuration:** Set up runtime and model parameters
2. **Construction:** Build the jaxfne model
3. **Simulation:** Run baseline and stimulus experiments
4. **Optimization:** Use AGSDR or other optimizers
5. **Visualization:** Generate spectrolaminar profiles
6. **Export:** Save artifacts as JSON with metadata

## Configuration

In [ ]:
SEED = 42
DURATION_MS = 1000.0
DT_MS = 0.1

def _default_spectrolaminar_config(areas=None, n_per_area=100, seed=None, duration_ms=1000.0, dt_ms=0.1):
    """Reproduces the removed jtfne.default_spectrolaminar_config (archived at
    jaxfne/configs/legacy/spectrolaminar_default.json) via the public Configuration
    fluent API. Verbatim from pre-removal source (commit 1715ec2^)."""
    areas = list(areas) if areas else ["V1", "V4"]
    cfg = (
        jtfne.Configuration()
        .runtime(seed=seed or 42, duration_ms=duration_ms, dt_ms=dt_ms, dtype='float32')
        .areas(areas)
    )
    for area in areas:
        cfg = cfg.column(area, layers=['L1', 'L2/3', 'L4', 'L5', 'L6'], n=n_per_area)
    layer_cts = {L: {'E': 0.75, 'PV': 0.1, 'SST': 0.08, 'VIP': 0.07}
                 for L in ['L1', 'L2/3', 'L4', 'L5', 'L6']}
    cfg = cfg.cell_types({'E': 0.75, 'PV': 0.10, 'SST': 0.08, 'VIP': 0.07})
    cfg = cfg.area_layer_cell_types(areas[0], layer_cts)
    if len(areas) > 1:
        cfg = cfg.area_layer_cell_types(areas[1], layer_cts)
    cfg = (
        cfg.uniform3d(radius_mm=0.25, height_mm=1.6)
        .connectivity(within_area='all_to_all_uniform_random', within_gain=0.35, edge_seed=seed or 42)
    )
    if len(areas) >= 2:
        cfg = cfg.inter_column_connectivity(
            source_area=areas[0], target_area=areas[1], mode='sparse',
            p_feedforward=0.3, p_feedback=0.2,
            feedforward_weight_range=(0.5, 2.0), feedback_weight_range=(0.3, 1.5),
        )
    cfg = (
        cfg.set_emitter('izhikevich', 'cortical_eig')
        .probes(['spikes', 'V_m', 'source', 'LFP', 'CSD', 'EEG', 'MEG', 'EMM'], n_contacts=16)
        .field(domain='laminar_column', conductivity='proxy', boundary='mean_zero_neumann')
        .objective(
            firing_rate_target={'E': 8.0, 'PV': 15.0, 'SST': 4.0, 'VIP': 2.0},
            band_definitions={'alpha_beta': (8.0, 25.0), 'gamma': (40.0, 150.0)},
        )
    )
    return cfg

cfg = _default_spectrolaminar_config(seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS)

## Workflow

Add your simulation, optimization, and visualization steps here.